# Supplementary Figure 1

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(ggvenn)
  library(patchwork)
  library(scales)
})

options(repr.plot.width = 11.94, repr.plot.height = 8.36)
class_colors <- c(Glycation = "#C44E52", Glycosylation = "#4C72B0")

theme_sf1 <- theme_classic(base_size = 12) +
  theme(
    panel.grid = element_blank(),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks = element_line(color = "black", linewidth = 0.35),
    axis.title = element_text(size = 12),
    axis.text = element_text(size = 10, color = "grey35"),
    legend.position = "top",
    legend.direction = "horizontal",
    legend.title = element_blank(),
    legend.text = element_text(size = 10),
    legend.key.height = unit(0.42, "cm"),
    legend.key.width = unit(0.42, "cm"),
    plot.margin = margin(4, 8, 4, 4)
  )

## Build and validate data

In [ ]:
qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')

rawGlycodata <- read.table("rosmap/glycoproteomics/dlpfcTissue/batch1/values/rawValues.txt", check.names = FALSE)

annotation_raw <- readRDS('rosmap/glycoproteomics/dlpfcTissue/batch1/annotation/gpAnnotation.Rds') %>%
  filter(!gpID %in% qc_remove_gpids) %>%
  # # the corrected file dropped Remove entirely
  mutate(Remove = FALSE) %>%
  mutate(GlycanTypeSummary = case_when(
    stringr::str_detect(GlycanType, "(^|;)gc(;|$)") & stringr::str_detect(GlycanType, "(^|;)N(;|$)") ~ "mixed",
    stringr::str_detect(GlycanType, "(^|;)gc(;|$)") ~ "glycation",
    stringr::str_detect(GlycanType, "(^|;)N(;|$)")  ~ "glycosylation",
    TRUE ~ GlycanTypeSummary
  )) %>%
  filter(gpID %in% rownames(rawGlycodata)) %>%
  mutate(glycopeptiform_id = paste(UniprotFirst, Sequence, GlycanComposition, GlycoSite, sep = "_"))

dir.create("figure_data", showWarnings = FALSE)
saveRDS(annotation_raw, file.path("figure_data", "Supplementary_Figure1_annotation.rds"))
dim(annotation_raw)

annotation_file <- file.path("figure_data", "Supplementary_Figure1_annotation.rds")
stopifnot(file.exists(annotation_file), file.info(annotation_file)$size > 0)
annotation <- readRDS(annotation_file)

# exclude sole-modification Trp C-mannosylation glycopeptiforms mislabeled as glycation
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
stopifnot(length(w_sole_exclude_gpids) == 54)
annotation <- annotation %>% filter(!gpID %in% w_sole_exclude_gpids)

required_columns <- c(
  "gpID", "UniprotFirst", "GlycanComposition", "GlycoSite",
  "BaseSequence", "GlycanTypeSummary", "Remove"
)
stopifnot(
  all(required_columns %in% names(annotation)),
  nrow(annotation) == 9044L,  # 9098 - 54 sole-modification Trp C-mannosylation gpIDs excluded above
  !anyDuplicated(annotation$gpID),
  all(!annotation$Remove),
  all(annotation$GlycanTypeSummary %in% c("glycation", "glycosylation", "mixed"))
)

annotation %>% count(GlycanTypeSummary, name = "Glycopeptiforms")

## Prepare panel data

In [ ]:
# Panels a-d: mixed entries carry both glycation and N-glycosylation
glycation_all <- annotation %>%
  filter(GlycanTypeSummary %in% c("glycation", "mixed"))
glycosylation_all <- annotation %>%
  filter(GlycanTypeSummary %in% c("glycosylation", "mixed"))

annotation_ab <- bind_rows(
  glycation_all %>% mutate(`Glycan Class` = "Glycation"),
  glycosylation_all %>% mutate(`Glycan Class` = "Glycosylation")
) %>%
  mutate(`Glycan Class` = factor(`Glycan Class`, levels = names(class_colors)))

distribution_data <- annotation_ab %>%
  group_by(`Glycan Class`, UniprotFirst) %>%
  summarise(
    Glycopeptiforms = n_distinct(gpID),
    GlycoSites = n_distinct(unlist(str_split(GlycoSite, ";"))),
    .groups = "drop"
  )

distribution_medians <- distribution_data %>%
  group_by(`Glycan Class`) %>%
  summarise(
    Glycopeptiforms = median(Glycopeptiforms),
    GlycoSites = median(GlycoSites),
    .groups = "drop"
  )

stopifnot(
  identical(as.numeric(distribution_medians$Glycopeptiforms), c(2, 3)),
  identical(as.numeric(distribution_medians$GlycoSites), c(2, 1))
)
distribution_medians

## Panel a: detected glycoproteins

In [ ]:
# Protein (UniprotFirst)-level Venn
venn_list <- list(
  Glycation = unique(glycation_all$UniprotFirst),
  Glycosylation = unique(glycosylation_all$UniprotFirst)
)

venn_counts <- tibble(
  Region = c("Glycation only", "Both", "Glycosylation only"),
  Count = c(
    length(setdiff(venn_list$Glycation, venn_list$Glycosylation)),
    length(intersect(venn_list$Glycation, venn_list$Glycosylation)),
    length(setdiff(venn_list$Glycosylation, venn_list$Glycation))
  )
)
stopifnot(identical(venn_counts$Count, c(455L, 129L, 633L)))
write.csv(venn_counts, file.path("figure_data", "Supplementary_Figure1_panel_a_counts.csv"), row.names = FALSE)

p_a <- ggvenn(
  venn_list,
  fill_color = unname(class_colors),
  fill_alpha = 0.60,
  stroke_size = 0.55,
  set_name_size = 4.2,
  text_size = 3.6,
  show_percentage = TRUE
) +
  theme(
    plot.margin = margin(18, 30, 10, 6),
    plot.tag.position = c(0.05, 0.985)
  )

venn_counts

## Panel b: total detected entities

In [ ]:
library(stringr)  # for str_split/str_extract in the Method 2 computations below

# Glycans uses Method 2 too (per chat discussion / checkGpAnnotation.R)
panel_b_counts <- annotation_ab %>%
  group_by(`Glycan Class`) %>%
  summarise(
    Glycans = n_distinct(unlist(str_split(GlycanComposition, ";"))),
    Glycoproteins = n_distinct(UniprotFirst),
    Glycopeptides = n_distinct(BaseSequence),
    Glycopeptiforms = n_distinct(gpID),
    .groups = "drop"
  )

site_residue <- annotation %>%
  select(gpID, GlycoSite) %>%
  mutate(GlycoSite = str_split(GlycoSite, ";")) %>%
  unnest(GlycoSite) %>%
  mutate(residue = str_extract(GlycoSite, "(?<=_)[A-Z]"))

glycosites_b <- tibble(
  `Glycan Class` = factor(c("Glycation", "Glycosylation"), levels = names(class_colors)),
  Glycosites = c(
    n_distinct(site_residue$GlycoSite[site_residue$residue == "K"]),
    n_distinct(site_residue$GlycoSite[site_residue$residue == "N"])
  )
)

panel_b_counts <- panel_b_counts %>%
  left_join(glycosites_b, by = "Glycan Class") %>%
  pivot_longer(-`Glycan Class`, names_to = "Feature", values_to = "Count") %>%
  filter(!(Feature == "Glycans" & `Glycan Class` == "Glycation")) %>%
  mutate(
    Feature = factor(
      Feature,
      levels = c("Glycans", "Glycoproteins", "Glycosites", "Glycopeptides", "Glycopeptiforms")
    ),
    Count_log10 = log10(Count),
    x_pos = case_when(
      Feature == "Glycans" ~ 1.5,
      `Glycan Class` == "Glycation" ~ 1,
      TRUE ~ 2
    )
  )

expected_b <- c(234L, 584L, 762L, 1564L, 1487L, 1573L, 1650L, 1582L, 7500L)
observed_b <- panel_b_counts %>% arrange(Feature, `Glycan Class`) %>% pull(Count)
stopifnot(identical(observed_b, expected_b))
write.csv(panel_b_counts, file.path("figure_data", "Supplementary_Figure1_panel_b_counts.csv"), row.names = FALSE)

p_b <- ggplot(panel_b_counts, aes(x = x_pos, y = Count_log10, fill = `Glycan Class`)) +
  geom_col(width = 0.58) +
  geom_text(aes(label = label_number(big.mark = ",", accuracy = 1)(Count)), vjust = 0.10, size = 3.2) +
  facet_wrap(~Feature, nrow = 1) +
  scale_x_continuous(breaks = NULL, expand = expansion(add = 0)) +
  scale_fill_manual(values = class_colors, name = NULL, drop = FALSE) +
  scale_y_continuous(limits = c(0, 4), breaks = 0:4, expand = expansion(mult = c(0, 0))) +
  coord_cartesian(xlim = c(0.45, 2.55), clip = "off") +
  labs(x = NULL, y = "Number detected (log10)") +
  theme_sf1 +
  theme(
    strip.background = element_blank(),
    strip.text = element_text(face = "bold", size = 8.5, margin = margin(b = 6)),
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    legend.position = "top",
    legend.margin = margin(0, 0, 2, 0),
    plot.margin = margin(2, 10, 10, 4),
    plot.tag.position = c(0.01, 0.985)
  )

panel_b_counts %>% select(`Glycan Class`, Feature, Count)

## Panels c-d: distributions per glycoprotein

In [ ]:
write.csv(
  distribution_data,
  file.path("figure_data", "Supplementary_Figure1_panel_cd_distributions.csv"),
  row.names = FALSE
)

distribution_legend <- guides(
  fill = guide_legend(title = NULL, order = 1),
  color = guide_legend(title = NULL, order = 1)
)

p_c <- ggplot(distribution_data, aes(x = Glycopeptiforms, fill = `Glycan Class`)) +
  geom_histogram(bins = 40, position = "dodge", alpha = 0.90, color = "white", linewidth = 0.18) +
  geom_vline(
    data = distribution_medians,
    aes(xintercept = Glycopeptiforms, color = `Glycan Class`),
    linetype = "dashed", linewidth = 0.90, show.legend = TRUE
  ) +
  scale_fill_manual(values = class_colors, name = NULL, drop = FALSE) +
  scale_color_manual(values = class_colors, name = NULL, drop = FALSE) +
  scale_x_continuous(breaks = seq(0, 250, 50), expand = expansion(mult = c(0.02, 0.02))) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.04))) +
  labs(x = "Glycopeptiforms per glycoprotein", y = "Glycoprotein count") +
  distribution_legend + theme_sf1 +
  theme(plot.margin = margin(2, 8, 4, 6), plot.tag.position = c(0.01, 0.985))

p_d <- ggplot(distribution_data, aes(x = GlycoSites, fill = `Glycan Class`)) +
  geom_histogram(bins = 40, position = "dodge", alpha = 0.90, color = "white", linewidth = 0.18) +
  geom_vline(
    data = distribution_medians,
    aes(xintercept = GlycoSites, color = `Glycan Class`),
    linetype = "dashed", linewidth = 0.90, show.legend = TRUE
  ) +
  scale_fill_manual(values = class_colors, name = NULL, drop = FALSE) +
  scale_color_manual(values = class_colors, name = NULL, drop = FALSE) +
  scale_x_continuous(breaks = seq(0, 40, 10), expand = expansion(mult = c(0.02, 0.02))) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.04))) +
  labs(x = "Glycosites per glycoprotein", y = "Glycoprotein count") +
  distribution_legend + theme_sf1 +
  theme(plot.margin = margin(2, 8, 4, 6), plot.tag.position = c(0.01, 0.985))

## Assemble and export

In [ ]:
top_row <- (p_a | p_b) + plot_layout(widths = c(0.80, 1.20))
bottom_row <- (p_c | p_d) + plot_layout(widths = c(1, 1))
supplementary_figure1 <- (top_row / bottom_row) +
  plot_layout(heights = c(1, 1)) +
  plot_annotation(tag_levels = "a") &
  theme(
    plot.tag = element_text(size = 14, face = "plain"),
    plot.background = element_rect(fill = "white", color = NA)
  )

ggsave(
  "Supplementary_Figure1.pdf", supplementary_figure1,
  width = 11.94, height = 8.36, units = "in", device = cairo_pdf, bg = "white"
)
stopifnot(
  file.exists("Supplementary_Figure1.pdf"),
  file.info("Supplementary_Figure1.pdf")$size > 0
)

supplementary_figure1